# Instal·lació de llibreries

In [2]:
#Install the necessary Python libraries
# import requests
#Install the necessary Python libraries
%pip install requests==2.32.4 -q --force-reinstall
%pip install langchain_community 
%pip install faiss-cpu -q
%pip install openai -q 
%pip install python-dotenv -q 
%pip install sentence_transformers -q 
%pip install torch -q 
%pip install transformers -q 

%pip install -q langchain
%pip install -U langchain-classic
%pip install -q langchain-huggingface
%pip install -q langchain-text-splitters


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated p

# make the necessary imports

In [3]:
#from langchain.document_loaders import TextLoader
#from langchain.vectorstores import FAISS
#from langchain.embeddings import HuggingFaceEmbeddings
#from langchain.text_splitter import RecursiveCharacterTextSplitter

#from langchain.chains import RetrievalQA

#from langchain.llms import HuggingFacePipeline
#from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import os

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

c:\Users\daniel.herrero.ext\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the document

In [4]:
def load_document(path):
    loader = TextLoader(path, encoding="utf-8")
    documents = loader.load()
    return documents

# Create embeddings with thenlper/gte-small

In [5]:
def prepare_knowledge_base(documents):
    # Split documents to create the chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    # Create chunk embeddings
    embedding_model = HuggingFaceEmbeddings(model_name="thenlper/gte-small")

    # Initialize FAISS vector store
    knowledge_base = FAISS.from_documents(chunks, embedding_model)
    len(chunks)
    return knowledge_base

# Create the LLM model sshleifer/tiny-gpt2

In [6]:
def create_llm():
    model_id = "sshleifer/tiny-gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)

    # Set pad token to avoid index out of range errors
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    llm_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=100,
        truncation=True,       # ← AÑADIR ESTA LÍNEA
        max_length=150,        # ← AÑADIR ESTA LÍNEA
        temperature=0.3,
        top_k=50,
        device=0 if torch.cuda.is_available() else -1,
    )
    return HuggingFacePipeline(pipeline=llm_pipeline)

# Create the QA chain

In [7]:
def build_chain(knowledge_base, llm):
    # Create the retrieval-based QA chain
    retriever = knowledge_base.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    return chain

# Terminal interface

In [8]:
def terminal_interface(qa_chain):
    print("System for questions about the document. Type 'exit' to finish.\n")
    while True:
        pregunta = input("make a question: ")
        if pregunta.lower() == "exit":
            break
        resposta = qa_chain.invoke(pregunta)
        print(f"\nanswer: {resposta['result']}\n")

# Execució General

In [9]:
if __name__ == "__main__":
    document_path = "Spider-Man.txt"

    if not os.path.exists(document_path):
        print(f"File '{document_path}' not found.")
    else:
        documents = load_document(document_path) # Load the document
        knowledge_base = prepare_knowledge_base(documents) # Split and index document
        llm = create_llm() # Create the language model
        chain = build_chain(knowledge_base, llm) # Build the QA chain
        terminal_interface(chain) # Start the terminal interface

C:\Users\daniel.herrero.ext\AppData\Local\Temp\ipykernel_22376\2576376488.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  return HuggingFacePipeline(pipeline=llm_pipeline)


System for questions about the document. Type 'exit' to finish.



Both `max_new_tokens` (=100) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

y debe tomarse la licencia de usar el término "peso" medido en kilogramos, y no en newtons, para acercarse al lector habitual.

se acerca al nivel superior o inferior también se consiguen resultados en alguna medida. Por ejemplo, entrenar al
87% o al 68% también puede producir hipertrofia, pero hacerlo al 100% o al 10% casi no producirá ese efecto.
*Nota: aunque el término correcto sea "masa", cuya unidad internacional es el kilogramo (kg), aquí utilizaremos el término "peso" por
ser el comúnmente manejado en este y otros deportes y así no confundir al lector no iniciado. La diferencia se comprende fácilmente si
se explica que la masa no cambia, pero el peso sí, en función de la gravedad existente. Como parece lógico, la medida de la gravedad
en la Tierra experimenta cambios relativamente pequeños independientemente

Both `max_new_tokens` (=100) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Cómo saber el peso adecuado y cuántos kilos perder
En primer lugar debe plantearse cuántos kilos quiere perder. Si el peso real es alto, es recomendable y mucho más práctico empezar con un objetivo modesto y factible. Por ejemplo, tratar de reducir entre un 5 y un 10 % del peso actual en no menos de 6 meses. Para una persona que pese 100 kg, la pérdida de peso en 6 meses no debe ser superior a 5‐10 kg. Para saber cuánto debe pesar, use la fórmula del Índice de Masa Corporal (IMC) (peso (kg) / talla x talla (m)). Un hombre de 90 kg de peso y 175 cm de altura tiene un IMC de 29.4 kg/m2. Para conseguir un IMC de 25 kg/m2 debe pesar 76.5 kg (le sobran 13.5 kg). Se estima apropiada una pérdida de peso de unos 400 g por semana. Por tanto, necesitará unos 9 meses para lograr el objetivo. 1

modo con esa carga.
Esta tabla t

Both `max_new_tokens` (=100) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



answer: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

PROGRESIONES DESDE CERO
AUSTRALIAN CHIN UPS
Dificultad: baja
Músculos implicados: espalda y bíceps
Progresiones: empezar barra alta e ir bajándola
Sirve para: dominadas supinas
Descripción: en barra baja, remo con agarre supino; se puede hacer con rodillas flexionadas o piernas rectas.

AUSTRALIAN PULL UPS
Dificultad: baja
Músculos implicados: espalda y bíceps
Progresiones: barra alta y ir bajando
Sirve para: dominadas pronas
Descripción: igual que chin ups pero agarre prono.

DEAD HANG
Dificultad: baja
Músculos implicados: antebrazos
Progresiones: australian pull ups
Sirve para: dominadas y retracciones escapulares
Descripción: colgarse de la barra y aguantar.

anterior y tríceps (largo).
Antagonistas: dorsal ancho, bíceps y pectoral (inferior).
21.2 ... agarre frontal
138
Variantes
Deltoides
Trapecio
Pectoral
Serr